In [1]:
from config import URL
from utils import q_run
from queries import CREATE_HNSW_INDEX

import millenniumdb_driver as mdb
import numpy as np
import time

In [2]:
q = '''
LET ?q = ?query_embedding
MATCH (?c :Embedding)
LET ?d = COSINE_DISTANCE(?q, ?c.value)
ORDER BY ?d
RETURN ?c, ?d
LIMIT ?k
'''

q_hnsw = '''
LET ?q = ?query_embedding
CALL HNSW_TOP_K("mi_indice", ?q, ?k, 1000)
  YIELD ?object AS ?c, ?distance AS ?d
RETURN ?c, ?d
'''

In [3]:
test_vector = np.array([0.029609, 0.036979, -0.020073, 0.018878, 0.018614, -0.003938, -0.011487, -0.047803, 0.028073, 0.026952, 0.007621, 0.008572, 0.101657, 0.044614, -0.047914, -0.033056, 0.009187, -0.019973, 0.072040, -0.031129, 0.018242, -0.035460, 0.078955, -0.013801, 0.045823, -0.027138, 0.018431, 0.040691, -0.028144, 0.060924, 0.021657, -0.018212, 0.044597, 0.067466, 0.051781, 0.020181, -0.041117, -0.040735, 0.019386, 0.015602, -0.015160, 0.024373, 0.032703, -0.008598, 0.018708, 0.001349, 0.069870, 0.031180, -0.029202, -0.021234, 0.015766, -0.014420, 0.030460, 0.023789, -0.043809, -0.010591, 0.010391, 0.060829, -0.051364, 0.043268, 0.011562, 0.038444, -0.014198, 0.016580, 0.036150, -0.003919, 0.007087, -0.018959, -0.050996, -0.012008, 0.029255, -0.038371, 0.039559, -0.001530, -0.021893, -0.006152, -0.045475, 0.006182, -0.007225, -0.023051, 0.060257, 0.005282, 0.040478, 0.042251, -0.003152, 0.036089, -0.025049, 0.024967, 0.016298, 0.031102, 0.017399, -0.055932, -0.038859, 0.023409, 0.003354, 0.005272, 0.016566, 0.034055, 0.028452, -0.058518, 0.038924, -0.071786, -0.009562, -0.088749, -0.083302, -0.044536, -0.045279, 0.010415, 0.043760, -0.023995, -0.054865, 0.010364, 0.022301, -0.018067, 0.005901, -0.042937, -0.002759, -0.057147, -0.001638, -0.050549, -0.015890, 0.028689, -0.023787, -0.028366, 0.011970, -0.001103, 0.005468, -0.042677, 0.005226, -0.061881, -0.033102, -0.021400, 0.009752, 0.007293, -0.029881, 0.061372, 0.023213, 0.037399, 0.012227, -0.006017, 0.034080, -0.060078, -0.034253, 0.005121, 0.043639, -0.072299, 0.019322, 0.022672, -0.018241, 0.015256, 0.051046, -0.023243, -0.024972, -0.038173, 0.042580, -0.008983, -0.035147, -0.040800, -0.045492, 0.035291, 0.042280, -0.001913, 0.007040, -0.011286, 0.066322, -0.004799, 0.022985, -0.003864, -0.057680, -0.023553, -0.007032, -0.009487, -0.067697, -0.000103, 0.029517, -0.032204, -0.005871, -0.064869, -0.031030, -0.065773, -0.028761, -0.038773, -0.024330, 0.039612, -0.016936, -0.029745, -0.009867, -0.028905, 0.013691, -0.001259, -0.023088, 0.014652, 0.038383, 0.032529, 0.031525, 0.016315, -0.008298, 0.011667, -0.025753, -0.045371, 0.006546, -0.010964, 0.032741, 0.061329, 0.011741, -0.059312, 0.002430, 0.054567, 0.044067, 0.043474, 0.009930, 0.035953, -0.041984, 0.042555, -0.025685, -0.013909, 0.002774, 0.013915, -0.049874, 0.013962, 0.008341, -0.055349, 0.029086, -0.006593, -0.025022, -0.010724, 0.017657, 0.000294, 0.027020, 0.013780, 0.047941, 0.031617, -0.042441, 0.035117, -0.030028, -0.011407, 0.018226, -0.005793, -0.033545, -0.135877, -0.005063, -0.000760, 0.004135, -0.030662, 0.005845, -0.065443, -0.037841, 0.055788, -0.089346, 0.005395, 0.022993, -0.048081, 0.009737, -0.049420, 0.013460, -0.014333, -0.065888, 0.044780, 0.042172, 0.052462, 0.004234, -0.021132, 0.011775, -0.020599, 0.021002, -0.009249, 0.062632, -0.016020, -0.030810, -0.053252, 0.014485, -0.001659, -0.067955, 0.053271, -0.015100, -0.029351, -0.035331, 0.047532, -0.035226, -0.010623, -0.006695, 0.070689, -0.054913, 0.013851, -0.035091, -0.022330, -0.050604, 0.006738, -0.077738, 0.009198, 0.060587, 0.057541, -0.005300, 0.007547, 0.070734, 0.035981, 0.054914, 0.022156, -0.017596, -0.000925, -0.022092, -0.033056, 0.044133, -0.055610, 0.003592, 0.019734, 0.098984, 0.020343, -0.051646, 0.024822, -0.042489, -0.006432, -0.080354, 0.058850, 0.007149, -0.056288, -0.012970, 0.069723, 0.054465, 0.014631, -0.010130, 0.069570, -0.034005, -0.019365, 0.064975, -0.043626, 0.047303, 0.014527, 0.060798, 0.019486, -0.006909, 0.049024, -0.013871, 0.052745, -0.082236, 0.025684, -0.050627, -0.045989, 0.076187, 0.014662, -0.025769, 0.028331, 0.009487, -0.009704, 0.021565, -0.017823, -0.020274, 0.004277, -0.023668, 0.046999, 0.011821, -0.085416, 0.032432, -0.018701, -0.049249, 0.028829, 0.031201, 0.000594, -0.040048, -0.023263, 0.000031, 0.043643, 0.002139, -0.012699, 0.011897, 0.014016, -0.008237, -0.017303, 0.025304, 0.016704, 0.067786, 0.037121, -0.066214, -0.011226, 0.041831, -0.032487, 0.032488, -0.026644, -0.035891, 0.020581, 0.050612, -0.001554, 0.018093, 0.001257, -0.029294, 0.040495, 0.029889, -0.054583, 0.013741, -0.001139, 0.045275, -0.026906, -0.029068, 0.011174, 0.041924, 0.014271, 0.034817, -0.006123, 0.033784, -0.048901, 0.041729, 0.039459, -0.015727, 0.040211, 0.020347, 0.039433, 0.000805, 0.037991, 0.014245, -0.015848, 0.086055, -0.009794, 0.033110, -0.025355, 0.009673, -0.039549, -0.009915, 0.034914, 0.015817, 0.027324, -0.022327, 0.007728, 0.003103, 0.018828, 0.042124, -0.062989, -0.008647, -0.016220, 0.017510, -0.045345, -0.005034, 0.023817, -0.038867, -0.005466, 0.060442, -0.057408, 0.020538, 0.026287, -0.029889, 0.034180, 0.001309, -0.038748, 0.042236, -0.029917, -0.016505, -0.017485, -0.049529, 0.005828, -0.001138, -0.011775, 0.002033, 0.037549, 0.029102, 0.007812, -0.011077, -0.030794, 0.059897, 0.002302, -0.007581, -0.130045, 0.052667, 0.038681, -0.014007, -0.001106, -0.014537, -0.029356, -0.003235, -0.064207, 0.045199, -0.045541, -0.044961, -0.017050, 0.038328, 0.013685, 0.018285, 0.063281, 0.033045, 0.061129, 0.051025, 0.014901, 0.018965, -0.012430, -0.017174, 0.044348, 0.003465, 0.028765, -0.067531, 0.034265, -0.005662, -0.050272, -0.002664, 0.030191, 0.019528, 0.007725, 0.012375, 0.055077, -0.048409, -0.043545, 0.042636, 0.000116, -0.012214, -0.013809, -0.045367, 0.016414, -0.044212, 0.019446, -0.041366, 0.005141, -0.011104, -0.019577, 0.073215, -0.049695, -0.011607, -0.018725, -0.013543, -0.003042, -0.031035, 0.054523, 0.011221, -0.041690, -0.009927, -0.014259, -0.018185, -0.050059, 0.044733, -0.031244, 0.014175, -0.031770, 0.039369, -0.032443, 0.034172, -0.029641, 0.055441, -0.078183, -0.062308, 0.015840, -0.013932, -0.049303, 0.030792, -0.019742, -0.024501, -0.034766, 0.028847, -0.007429, 0.023198, -0.028163, 0.030010, -0.013450, -0.001650, -0.023353, 0.022669, -0.027136, 0.011373, 0.038290, -0.037404, 0.040887, 0.027979, -0.001396, -0.003545, -0.020977, -0.017722, -0.030863, -0.028532, 0.048709, -0.028173, 0.046452, 0.038222, -0.000020, 0.000662, 0.072478, -0.038529, 0.023932, 0.011675, -0.007224, 0.028770, 0.017725, -0.030048, -0.018820, 0.076883, -0.042018, -0.063293, -0.032197, -0.036170, 0.041910, 0.020261, 0.038688, 0.004570, -0.009452, -0.180239, 0.025346, -0.015926, -0.050548, 0.018123, -0.013962, 0.012437, 0.021952, 0.029233, 0.028045, 0.036610, 0.001413, 0.024054, -0.023282, 0.018569, 0.007083, -0.014250, 0.000202, -0.010661, 0.061566, -0.024691, 0.075681, -0.043215, -0.035234, -0.003794, -0.014195, -0.009484, 0.011415, 0.021962, 0.019296, 0.022449, -0.011323, 0.024384, 0.047080, -0.041663, -0.022606, 0.000051, 0.020974, -0.114325, 0.033277, -0.009641, -0.039238, -0.035383, 0.000634, 0.010256, 0.047924, 0.031455, 0.028944, -0.041475, -0.029742, 0.042709, 0.019128, -0.063747, 0.000454, 0.010961, 0.020644, 0.011731, 0.035573, 0.038657, -0.027588, 0.012432, 0.030690, -0.039577, 0.038926, -0.023629, 0.009782, 0.031806, -0.017256, -0.021822, -0.016629, 0.005479, -0.017038, -0.026520, 0.003777, 0.057338, -0.035050, 0.023416, -0.026774, -0.067925, 0.033199, -0.048284, -0.043705, -0.015463, 0.024737, -0.028178, -0.006375, -0.046527, -0.012638, 0.003325, 0.042020, -0.033926, 0.003214, -0.008937, 0.000183, -0.000175, -0.051155, -0.014565, 0.020239, -0.021888, -0.015155, -0.029439, -0.027135, -0.046954, 0.033657, 0.066581, -0.035591, -0.096523, 0.020405, -0.044575, -0.024968, 0.018017, 0.048762, 0.032528, -0.011172, 0.014119, -0.013419, 0.030168, -0.032241, 0.004632, -0.059165, -0.023014, 0.004511, 0.007542, -0.006986, 0.034955, -0.034914, -0.012960, 0.018871, 0.026240, 0.010362, -0.055137, -0.037079, 0.014958, -0.011276, -0.025719, 0.030852, 0.039581, -0.019793, 0.035387, 0.045809, 0.082167, -0.076580, -0.018066, -0.020906, -0.010003, 0.015604, 0.047118, -0.026781, 0.009463, -0.002828, -0.037576, -0.078642, -0.024858, -0.017144, -0.003119, 0.062162, 0.019706, 0.009170, 0.007695, 0.023199, 0.019785, -0.026823, 0.046008, 0.017729, 0.021242, -0.026515, -0.040155, -0.013543, 0.029462, 0.014994, -0.008466, -0.013837, 0.021982, 0.022500, 0.021916, 0.008757, 0.061475, -0.028695, -0.040445, 0.051484])

K = 10000
N = 50

In [4]:
# SIN INDICE HNSW

driver = mdb.driver(URL)
session = driver.session()

tiempos = []
for _ in range(N):
    inicio = time.perf_counter()
    result = q_run(
        session=session,
        query=q,
        parameters={"query_embedding": test_vector},
        replaces={"k": K},
    )
    tiempos.append(time.perf_counter() - inicio)

session.close()
driver.close()

print(f"N = {N}")
print(f"Tiempo promedio: {sum(tiempos) / len(tiempos):.6f} s")
print(f"Tiempo mínimo: {min(tiempos):.6f} s")
print(f"Tiempo máximo: {max(tiempos):.6f} s")

N = 50
Tiempo promedio: 0.672899 s
Tiempo mínimo: 0.624906 s
Tiempo máximo: 1.002665 s


In [5]:
# CREAR INDICE HNSW

driver = mdb.driver(URL)
session = driver.session()

tiempo = time.perf_counter()
result = q_run(
    session=session, 
    query=CREATE_HNSW_INDEX
)
tiempo = time.perf_counter() - tiempo

session.close()
driver.close()

print(f"Tiempo: {tiempo:.6f} s")

Tiempo: 31.880023 s


In [6]:
# CON INDICE HNSW

driver = mdb.driver(URL)
session = driver.session()

tiempos = []
for _ in range(N):
    inicio = time.perf_counter()
    result = q_run(
        session=session,
        query=q,
        parameters={"query_embedding": test_vector},
        replaces={"k": K},
    )
    tiempos.append(time.perf_counter() - inicio)

session.close()
driver.close()

print(f"N = {N}")
print(f"Tiempo promedio: {sum(tiempos) / len(tiempos):.6f} s")
print(f"Tiempo mínimo: {min(tiempos):.6f} s")
print(f"Tiempo máximo: {max(tiempos):.6f} s")

N = 50
Tiempo promedio: 0.689266 s
Tiempo mínimo: 0.641672 s
Tiempo máximo: 0.989690 s


In [7]:
# CON INDICE HNSW + METODO APROXIMADO CALL HNSW_TOP_K

driver = mdb.driver(URL)
session = driver.session()

tiempos = []
for _ in range(N):
    inicio = time.perf_counter()
    result = q_run(
        session=session,
        query=q_hnsw,
        parameters={"query_embedding": test_vector},
        replaces={"k": K},
    )
    tiempos.append(time.perf_counter() - inicio)

session.close()
driver.close()

print(f"N = {N}")
print(f"Tiempo promedio: {sum(tiempos) / len(tiempos):.6f} s")
print(f"Tiempo mínimo: {min(tiempos):.6f} s")
print(f"Tiempo máximo: {max(tiempos):.6f} s")

N = 50
Tiempo promedio: 0.022479 s
Tiempo mínimo: 0.015727 s
Tiempo máximo: 0.087856 s
